In [2]:
import hashlib
import re
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


URL = "https://www.scouting.org/skills/merit-badges/all/"
BASE = "https://www.scouting.org"
OUTPUT_XLSX = "bsa_merit_badges.xlsx"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
    )
}

REGION_FIXED = "America"


def norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()


def fetch_html(url: str) -> str:
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text


def importance_1_to_100(name: str) -> int:
    """
    Deterministic importance score in [1..100] based on the badge name.
    Not official—just a stable ranking.
    """
    h = hashlib.sha256(name.lower().encode("utf-8")).hexdigest()
    n = int(h[:8], 16)  # take first 32 bits
    return (n % 100) + 1


def scrape_merit_badges(url: str = URL) -> pd.DataFrame:
    html = fetch_html(url)
    soup = BeautifulSoup(html, "html.parser")

    cards = soup.select("div.mb-card")
    if not cards:
        # If this happens, it can mean the list is rendered by JS (then you need Selenium).
        return pd.DataFrame(columns=["badge_name", "importance", "region"])

    rows = []
    seen = set()

    for card in cards:
        a = card.select_one("h2.mb-card-title a")
        if not a:
            continue

        name = norm_text(a.get_text())
        href = a.get("href", "")
        full_url = urljoin(BASE, href)

        if not name:
            continue
        key = name.lower()
        if key in seen:
            continue
        seen.add(key)

        rows.append(
            {
                "badge_name": name,
                "importance": importance_1_to_100(name),
                "region": REGION_FIXED,
                "url": full_url,  # optional (not requested, but useful)
            }
        )

    df = pd.DataFrame(rows)

    # If you truly only want the 3 columns, uncomment the next line:
    # df = df[["badge_name", "importance", "region"]]

    return df


def save_to_excel(df: pd.DataFrame, path: str = OUTPUT_XLSX):
    if df.empty:
        print("No badges found; nothing to save.")
        return
    # Save only requested columns (name/date/description equivalent is name/importance/region here)
    out = df[["badge_name", "importance", "region"]]
    out.to_excel(path, index=False)
    print(f"Saved {len(out)} rows to {path}")


if __name__ == "__main__":
    df = scrape_merit_badges()
    print("Rows:", len(df))
    print(df.head(10))
    save_to_excel(df)

Rows: 141
                badge_name  importance   region  \
0        American Business          31  America   
1        American Cultures          49  America   
2        American Heritage           8  America   
3  American Indian Culture          56  America   
4           American Labor           8  America   
5           Animal Science          95  America   
6                Animation          13  America   
7              Archaeology          64  America   
8                  Archery          88  America   
9             Architecture          16  America   

                                                 url  
0  https://www.scouting.org/merit-badges/american...  
1  https://www.scouting.org/merit-badges/american...  
2  https://www.scouting.org/merit-badges/american...  
3  https://www.scouting.org/merit-badges/american...  
4  https://www.scouting.org/merit-badges/american...  
5  https://www.scouting.org/merit-badges/animal-s...  
6   https://www.scouting.org/merit-badges/a

In [4]:
import json
import re
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


LIST_URL = "https://www.scouting.org/partnerwithus/partners/"
BASE = "https://www.scouting.org"
OUTPUT_XLSX = "bsa_partners.xlsx"
REGION_FIXED = "America"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
    )
}


def norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()


def fetch_soup(url: str) -> BeautifulSoup:
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")


def extract_partner_name(detail_soup: BeautifulSoup) -> str:
    # Most partner pages should have an H1 title somewhere
    for sel in ["h1.entry-title", ".entry-title", "header h1", "main h1", "h1"]:
        el = detail_soup.select_one(sel)
        if el:
            name = norm_text(el.get_text())
            if name:
                return name

    # Meta fallback
    og = detail_soup.find("meta", attrs={"property": "og:title"})
    if og and og.get("content"):
        return norm_text(og["content"])

    title_tag = detail_soup.find("title")
    return norm_text(title_tag.get_text()) if title_tag else ""


def scrape_partners() -> pd.DataFrame:
    list_soup = fetch_soup(LIST_URL)

    cards = list_soup.select('div.pp-wrapper-link[data-pp-wrapper-link]')
    if not cards:
        print("No partner cards found with selector div.pp-wrapper-link[data-pp-wrapper-link].")
        print("This may mean the page is JS-rendered for you; use Selenium if needed.")
        return pd.DataFrame(columns=["partner_name", "region"])

    # Extract detail URLs from the JSON inside data-pp-wrapper-link
    detail_urls = []
    seen = set()

    for card in cards:
        raw = card.get("data-pp-wrapper-link", "")
        if not raw:
            continue

        # raw is JSON-like; BeautifulSoup already unescapes &quot; into "
        try:
            data = json.loads(raw)
        except Exception:
            # ultra-safe fallback: try to pull "/partnerwithus/partners/..." via regex
            m = re.search(r'("url"\s*:\s*")([^"]+)(")', raw)
            if not m:
                continue
            data = {"url": m.group(2)}

        href = data.get("url", "")
        if not href:
            continue

        full = urljoin(BASE, href)
        key = full.rstrip("/")
        if key in seen:
            continue
        seen.add(key)
        detail_urls.append(full)

    rows = []
    for detail_url in detail_urls:
        try:
            detail_soup = fetch_soup(detail_url)
        except Exception as e:
            print(f"Failed to fetch detail {detail_url}: {e}")
            continue

        name = extract_partner_name(detail_soup)
        if not name:
            print(f"Could not extract name from {detail_url}")
            continue

        rows.append(
            {
                "partner_name": name,
                "region": REGION_FIXED,
            }
        )

    df = pd.DataFrame(rows)
    return df[["partner_name", "region"]] if not df.empty else df


def save_to_excel(df: pd.DataFrame, path: str = OUTPUT_XLSX):
    if df.empty:
        print("No partners found; nothing to save.")
        return
    df.to_excel(path, index=False)
    print(f"Saved {len(df)} rows to {path}")


if __name__ == "__main__":
    df = scrape_partners()
    print("Rows:", len(df))
    print(df.head(10))
    save_to_excel(df)

Rows: 38
                        partner_name   region
0             9/11 Memorial & Museum  America
1                          AB Emblem  America
2                ALPS Mountaineering  America
3    American Numismatic Association  America
4                 American Red Cross  America
5                 Black Pug Software  America
6  Blue Origin | Club For The Future  America
7                   Build Submarines  America
8                             ClassB  America
9                           CleanHub  America
Saved 38 rows to bsa_partners.xlsx
